# Module 13 lab · Fine-tuning

The code from every lesson of [this module](https://aicourses.dev/research/fine-tuning), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**First time?** Read the [one-time setup](https://aicourses.dev/research/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 13.2 Prompt first: in-context learning, few-shot examples and when to fine-tune

Many problems are solved by a better prompt. Knowing what prompting cannot fix, measured with your own eval, is how you decide a fine-tune is worth its cost.

[Read the lesson](https://aicourses.dev/research/fine-tuning/prompting-first)

### Zero, one, few

You met “5-shot” in lesson 12.3: it is the same idea, used to test base models. Here is a prompt builder for labelling café reviews, with `k` solved examples:

In [ ]:
examples = [
    ("The dosa was crisp and the chutney fresh.", "positive"),
    ("Waited 40 minutes for cold coffee.", "negative"),
    ("Friendly staff, but the bread was stale.", "negative"),
]

def prompt(review, k):           # k = how many solved examples
    lines = ["Label each café review as positive or negative.", ""]
    for text, label in examples[:k]:
        lines += [f"Review: {text}", f"Label: {label}", ""]
    lines += [f"Review: {review}", "Label:"]
    return "\n".join(lines)

print(prompt("Best filter coffee in town!", 2))

*The lesson shows:*

```text
Label each café review as positive or negative.

Review: The dosa was crisp and the chutney fresh.
Label: positive

Review: Waited 40 minutes for cold coffee.
Label: negative

Review: Best filter coffee in town!
Label:
```

### When prompting runs out

The arithmetic for the second point is simple enough to check (`:,` writes a number with commas):

In [ ]:
calls = 1_000_000                 # requests a month
short, long = 40, 640             # prompt tokens: instruction only,
                                  # or instruction plus 12 examples
extra = (long - short) * calls
print(f"{extra:,} extra prompt tokens a month")
# 600,000,000 extra prompt tokens a month
print(long / short)               # 16.0

## 13.3 Supervised fine-tuning with Hugging Face: chat templates and loss masking

Supervised fine-tuning (SFT) trains a model on example conversations. A chat template marks who is speaking, and loss masking trains only on the assistant’s replies. Get either one wrong and the model quietly gets worse.

[Read the lesson](https://aicourses.dev/research/fine-tuning/sft-and-chat-templates)

### The chat template

A chat model sees a conversation as a list of messages, each with a **role** (system, user or assistant) and some **content**. The template turns that list into one string, using special tokens the tokenizer of lesson 10.8 keeps whole. Qwen2.5’s published template wraps every turn in `<|im_start|>` and `<|im_end|>`, and adds a default system message when you give none. Here it is rebuilt by hand:

In [ ]:
# Qwen2.5's default system message, from its chat template
SYSTEM = ("You are Qwen, created by Alibaba Cloud. "
          "You are a helpful assistant.")

def qwen_chat(messages, add_generation_prompt=False):
    text = f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
    for m in messages:
        text += f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n"
    if add_generation_prompt:      # open the door for a reply
        text += "<|im_start|>assistant\n"
    return text

chat = [
    {"role": "user", "content": "Is the rye bread vegan?"},
    {"role": "assistant", "content": "Yes, it has no milk or eggs."},
]
print(qwen_chat(chat))

*The lesson shows:*

```text
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Is the rye bread vegan?<|im_end|>
<|im_start|>assistant
Yes, it has no milk or eggs.<|im_end|>
```

### Loss masking: learn the replies, not the questions

The fix is a **loss mask**: you mark the positions that should not count. In PyTorch, a target of −100 is skipped by `cross_entropy`, whose `ignore_index` is −100 by default. Below, eight token ids stand in for one conversation, the last three being the assistant’s reply, and random numbers stand in for a model’s logits:

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
ids = torch.tensor([5, 9, 2, 7, 7, 3, 8, 1])  # one conversation
reply_starts = 5              # tokens 5 onwards are the assistant's
logits = torch.randn(len(ids), 10)  # a pretend model, vocabulary of 10

# position t predicts token t + 1: shift by one
pred, target = logits[:-1], ids[1:].clone()
print(F.cross_entropy(pred, target))          # tensor(3.0459)

target[: reply_starts - 1] = -100             # hide the prompt
print(target)  # tensor([-100, -100, -100, -100,    3,    8,    1])
print(F.cross_entropy(pred, target))          # tensor(3.2938)
print(F.cross_entropy(pred[reply_starts - 1:], ids[reply_starts:]))
# tensor(3.2938)

### SFT with TRL

**TRL** is Hugging Face’s library for post-training, the training that comes after pretraining. Its `SFTTrainer` applies the chat template and the loss mask for you. Give it a **prompt-completion** dataset, where each row holds the conversation so far and the reply to learn, and it trains on the completion only by default. Hugging Face’s `datasets` library holds the rows:

The lesson installs this first:

In [ ]:
!pip install trl

This cell needs a GPU runtime.

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

def row(question, answer):
    return {"prompt": [{"role": "user", "content": question}],
            "completion": [{"role": "assistant", "content": answer}]}

data = Dataset.from_list([
    row("Is the rye bread vegan?", "Yes, it has no milk or eggs."),
    row("Do you open on Sundays?", "Yes, from 8 am to 2 pm."),
])
trainer = SFTTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    args=SFTConfig(output_dir="sft-bakery", num_train_epochs=3,
                   fp16=True),       # the T4 has no BF16
    train_dataset=data,
)
trainer.train()

## 13.4 Rank: building a big matrix from two thin ones

Many big grids of numbers can be written, nearly, as a tall thin matrix times a wide thin one, which needs far fewer numbers. This idea, called low rank, is what makes LoRA cheap.

[Read the lesson](https://aicourses.dev/research/fine-tuning/rank)

### One column times one row

Multiply a column of 3 numbers by a row of 3 numbers, a 3-by-1 matrix times a 1-by-3 one, and you get a full 3-by-3 grid:

In [ ]:
import numpy as np

col = np.array([[1], [2], [3]])     # a 3 x 1 column
row = np.array([[2, 0, 1]])         # a 1 x 3 row
print(col @ row)
# [[2 0 1]
#  [4 0 2]
#  [6 0 3]]

### The worked count

Take a 4,096-by-4,096 matrix, the size of one attention projection in many 7-billion-parameter models:

In [ ]:
d = 4096
print(f"full: {d * d:,}")           # full: 16,777,216
for r in (1, 8, 64):
    thin = d * r + r * d            # a d x r matrix and an r x d one
    print(f"rank {r}: {thin:,} numbers, {d * d // thin}x fewer")
# rank 1: 8,192 numbers, 2048x fewer
# rank 8: 65,536 numbers, 256x fewer
# rank 64: 524,288 numbers, 32x fewer

### Nearly low rank: the SVD

Here are the pizza ratings. Each friend gets a random love of spicy and of sweet, each topping a spiciness and a sweetness, and a little noise is added on top:

In [ ]:
rng = np.random.default_rng(0)
taste = rng.uniform(0, 1, size=(6, 2))   # each friend: spicy, sweet
# chilli, jalapeño, pineapple, honey, mushroom: how spicy, how sweet
toppings = np.array([[9, 8, 1, 2, 5], [1, 2, 9, 8, 5]])
ratings = taste @ toppings + rng.normal(0, 0.3, size=(6, 5))

U, S, Vt = np.linalg.svd(ratings)
print(S.round(2))               # [32.26  7.01  0.55  0.39  0.1 ]
for k in (1, 2, 3):
    approx = U[:, :k] * S[:k] @ Vt[:k]    # best rank-k version
    print(k, np.abs(ratings - approx).mean().round(3))
# 1 0.916
# 2 0.095
# 3 0.052

## 13.5 LoRA: fine-tuning a low-rank update

LoRA leaves the model’s weights alone and trains a small low-rank update beside them. For most post-training it matches full fine-tuning, and it fits on a free GPU.

[Read the lesson](https://aicourses.dev/research/fine-tuning/lora)

### A LoRA layer from scratch

In [ ]:
import torch

class LoRALinear(torch.nn.Module):
    def __init__(self, base, r=4, alpha=8):
        super().__init__()
        self.base = base.requires_grad_(False)   # frozen original
        d_out, d_in = base.weight.shape
        self.A = torch.nn.Parameter(torch.randn(r, d_in) * 0.01)
        self.B = torch.nn.Parameter(torch.zeros(d_out, r))
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale

torch.manual_seed(0)
base = torch.nn.Linear(256, 256)
layer = LoRALinear(base)
x = torch.randn(32, 256)
print(torch.equal(layer(x), base(x)))     # True: B starts at zero
trainable = sum(p.numel() for p in layer.parameters()
                if p.requires_grad)
print(trainable, base.weight.numel())     # 2048 65536

Now test it. Below, a hidden low-rank change is added to a layer’s outputs, and the adapter is trained to learn it:

In [ ]:
def fit(change_rank, r, steps=600):
    torch.manual_seed(0)
    base = torch.nn.Linear(256, 256)
    layer = LoRALinear(base, r=r, alpha=2 * r)
    # the new job: the old layer plus a hidden low-rank change
    U, V = torch.randn(256, change_rank), torch.randn(change_rank, 256)
    X = torch.randn(2048, 256)
    with torch.no_grad():
        Y = base(X) + X @ (0.05 * U @ V).T
    opt = torch.optim.Adam([layer.A, layer.B], lr=1e-2)
    for step in range(steps):
        i = torch.randint(0, 2048, (64,))
        loss = ((layer(X[i]) - Y[i]) ** 2).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    with torch.no_grad():
        return ((layer(X) - Y) ** 2).mean().item(), layer

for change_rank, r in [(2, 4), (16, 4), (16, 16)]:
    err, layer = fit(change_rank, r)
    print(f"change rank {change_rank}, LoRA rank {r}: {err:.2g}")
# change rank 2, LoRA rank 4: 2.8e-06
# change rank 16, LoRA rank 4: 6.5
# change rank 16, LoRA rank 16: 1.3e-08

`:.2g` keeps two significant figures, so 2.8e-06 is 0.0000028. A rank-4 adapter learns a rank-2 change almost perfectly, fails on a rank-16 one, and a rank-16 adapter fixes that. Finally, merge the last adapter and check nothing changed:

In [ ]:
W = layer.base.weight + layer.scale * layer.B @ layer.A  # merge
x = torch.randn(8, 256)
print(torch.allclose(x @ W.T + layer.base.bias, layer(x), atol=1e-5))
# True

### LoRA with PEFT

**PEFT**, Hugging Face’s library for parameter-efficient fine-tuning, training only a few extra numbers, wraps any Transformers model:

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B")
config = LoraConfig(r=16, lora_alpha=32, target_modules="all-linear",
                    task_type="CAUSAL_LM")
model = get_peft_model(model, config)
model.print_trainable_parameters()

## 13.6 QLoRA: fitting bigger models on small GPUs

QLoRA stores the frozen model in 4 bits instead of 16 and trains a LoRA beside it. Real models then fit on free or cheap GPUs, so you can do the work, not only read about it.

[Read the lesson](https://aicourses.dev/research/fine-tuning/qlora)

### Four bits, sixteen levels

The memory sum is simple:

In [ ]:
billion = 1e9
bf16 = billion * 16 / 8 / 1e9           # 16 bits = 2 bytes each
nf4 = billion * 4 / 8 / 1e9             # 4 bits = half a byte each
consts = billion * 0.127 / 8 / 1e9      # QLoRA's block constants
print(bf16, nf4, round(nf4 + consts, 3))  # 2.0 0.5 0.516
print(7 * bf16, round(7 * (nf4 + consts), 2))  # 14.0 3.61

### Levels where the weights are

Here is a small experiment. It makes 64,000 bell-shaped weights, then quantizes them two ways: with 16 evenly spaced levels, and with 16 levels spaced like a bell curve, each level holding about the same share of a bell’s values. `NormalDist().inv_cdf(p)`, from Python’s own `statistics` module, finds the point of a standard bell with a share $p$ of its values below it: 0.5 gives 0, the middle, and 0.975 gives 1.96, the number from lesson 12.4. This is a simplified cousin of NF4, not the exact table QLoRA uses.

In [ ]:
import numpy as np
from statistics import NormalDist

rng = np.random.default_rng(0)
w = rng.normal(0, 0.02, size=64 * 1000)   # bell-shaped weights

def quantize(w, levels, block=64):
    blocks = w.reshape(-1, block)
    scale = np.abs(blocks).max(axis=1, keepdims=True)  # per block
    x = blocks / scale                          # now in [-1, 1]
    nearest = np.abs(x[..., None] - levels).argmin(axis=-1)
    return (levels[nearest] * scale).ravel()    # back again

even = np.linspace(-1, 1, 16)           # 16 evenly spaced levels
bell = np.array([NormalDist().inv_cdf((i + 0.5) / 16)
                 for i in range(16)])
bell /= np.abs(bell).max()              # 16 levels, bell-spaced
for name, levels in (("even", even), ("bell", bell)):
    err = np.abs(quantize(w, levels) - w).mean()
    print(name, err / np.abs(w).mean())  # error per unit of size
# even 0.10729010766651395
# bell 0.09098592028411288

### Running QLoRA

The **bitsandbytes** library does the 4-bit storage. It is built mainly for NVIDIA GPUs, so run this on a Colab GPU runtime, after `!pip install trl peft bitsandbytes`:

The lesson installs this first:

In [ ]:
!pip install trl peft bitsandbytes

This cell needs a GPU runtime.

In [ ]:
import torch
from peft import LoraConfig
from transformers import BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

quant = BitsAndBytesConfig(load_in_4bit=True,
                           bnb_4bit_quant_type="nf4",
                           bnb_4bit_use_double_quant=True,
                           bnb_4bit_compute_dtype=torch.float16)
trainer = SFTTrainer(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    args=SFTConfig(output_dir="qlora-bakery", learning_rate=2e-4,
                   fp16=True),
    train_dataset=data,         # your dataset from lesson 13.3
    quantization_config=quant,
    peft_config=LoraConfig(r=16, lora_alpha=32,
                           target_modules="all-linear"),
)
trainer.train()

## 13.7 Building a fine-tuning dataset from your own data

Building the data, deduplicating it and decontaminating it against your eval is where much of a fine-tune’s time goes.

[Read the lesson](https://aicourses.dev/research/fine-tuning/fine-tuning-data)

### Deduplicate: give every example a fair weight

A row that appears 300 times is trained on 300 times, so the model leans towards it. First **normalise** the text: lower case, punctuation removed, spaces squeezed. Then compute a **hash** of each row, a short fingerprint that is the same for identical text, and keep only the first row with each fingerprint:

In [ ]:
import hashlib
import re

def normalise(text):              # lower case, no punctuation, 1 space
    return " ".join(re.sub(r"[^\w\s]", " ", text.lower()).split())

rows = ["Do you deliver on Sundays?", "do you  deliver on SUNDAYS",
        "Is the rye bread vegan?", "Do you deliver on Sunday?"]
seen, kept = set(), []
for row in rows:
    key = hashlib.sha256(normalise(row).encode()).hexdigest()
    if key not in seen:
        seen.add(key)
        kept.append(row)
print(kept)
# ['Do you deliver on Sundays?', 'Is the rye bread vegan?',
#  'Do you deliver on Sunday?']

`re.sub(r"[^\w\s]", " ", ...)` swaps every character that is not a letter, a digit or a space for a space. `hashlib.sha256` is a standard fingerprint maker; `.encode()` turns the text into bytes first, and `.hexdigest()` writes the fingerprint as text. The second row became identical to the first after normalising, so it went. “Sunday” against “Sundays” survives, because exact matching cannot see that they are nearly the same. For that, compare **n-grams**, runs of $n$ consecutive words, and measure the share two rows have in common:

In [ ]:
def ngrams(text, n):
    words = normalise(text).split()
    return {tuple(words[i:i + n]) for i in range(len(words) - n + 1)}

def similarity(a, b, n=3):        # shared n-grams over all n-grams
    sa, sb = ngrams(a, n), ngrams(b, n)
    return len(sa & sb) / len(sa | sb)

print(similarity("Do you deliver on Sundays?",
                 "Do you deliver on Sunday?"))   # 0.5

### Decontaminate: keep your eval honest

Your eval from Module 12 is the judge of this fine-tune. If its questions are in the training data, the fine-tune can score well by remembering, and the gain you report is partly fake. So check every training row against every eval item. The GPT-3 team flagged test items that shared any 13-word run with their training data; for short questions, a smaller $n$ such as 8 catches more.

In [ ]:
eval_items = [
    "A baker makes 12 loaves a day for 5 days. How many loaves?",
]
train = [
    "Our shop opens at 8 am on weekdays.",
    "A baker makes 12 loaves a day for 5 days. How many in total?",
    "How many loaves does a baker make in 5 days at 12 a day?",
]
banned = set().union(*(ngrams(item, 8) for item in eval_items))
for text in train:
    hits = len(ngrams(text, 8) & banned)
    print(hits, "DROP" if hits else "keep", text)
# 0 keep Our shop opens at 8 am on weekdays.
# 5 DROP A baker makes 12 loaves a day for 5 days. How many in total?
# 0 keep How many loaves does a baker make in 5 days at 12 a day?

## 13.8 Preference data and DPO

It is often easier to say which of two answers is better than to write the best one. Direct preference optimization (DPO) trains on such pairs, and the KL from Module 9 keeps the model close to where it started.

[Read the lesson](https://aicourses.dev/research/fine-tuning/dpo)

### The idea: push the pair apart, relative to where you started

In [ ]:
import torch
import torch.nn.functional as F

def dpo_loss(pi_w, ref_w, pi_l, ref_l, beta=0.1):
    """Log-probs of the chosen (w) and rejected (l) answers."""
    margin = (pi_w - ref_w) - (pi_l - ref_l)
    return -F.logsigmoid(beta * margin)

t = torch.tensor
print(dpo_loss(t(-13.0), t(-13.0), t(-14.0), t(-14.0)))  # 0.6931
print(dpo_loss(t(-12.0), t(-13.0), t(-15.0), t(-14.0)))  # 0.5981

### β is the leash

Real preferences are noisy: of 100 raters, 60 might prefer one answer and 40 the other. Below, one prompt has three possible answers and votes from raters, and a tiny policy, just three logits, is trained with DPO at two values of β, starting from the reference of one third each:

In [ ]:
answers = ["short, right", "long, right", "wrong"]
# (chosen, rejected, how many raters preferred it that way)
votes = [(0, 1, 60), (1, 0, 40), (0, 2, 90), (2, 0, 10),
         (1, 2, 85), (2, 1, 15)]
w, l, n = (torch.tensor(c) for c in zip(*votes))
ref = torch.log_softmax(torch.zeros(3), dim=0)    # start: 1/3 each

for beta in (0.1, 1.0):
    logits = torch.zeros(3, requires_grad=True)    # the policy
    opt = torch.optim.Adam([logits], lr=0.05)
    for step in range(3000):
        pi = torch.log_softmax(logits, dim=0)
        margin = (pi[w] - ref[w]) - (pi[l] - ref[l])
        loss = -(n * F.logsigmoid(beta * margin)).sum() / n.sum()
        opt.zero_grad()
        loss.backward()
        opt.step()
    p = pi.exp().detach()
    kl = (p * (pi.detach() - ref)).sum()
    print(beta, p.numpy().round(3), round(kl.item(), 3))
# 0.1 [0.985 0.015 0.   ] 1.019
# 1.0 [0.564 0.372 0.064] 0.231

### DPO with TRL

This cell needs a GPU runtime.

In [ ]:
from datasets import Dataset
from trl import DPOConfig, DPOTrainer

pairs = Dataset.from_list([{
    "prompt": [{"role": "user", "content": "Can I order a cake?"}],
    "chosen": [{"role": "assistant", "content": "Yes! Which one?"}],
    "rejected": [{"role": "assistant", "content": "Cakes are old."}],
}])
trainer = DPOTrainer(model="sft-bakery",       # your SFT checkpoint
                     args=DPOConfig(output_dir="dpo-bakery", beta=0.1,
                                    fp16=True),
                     train_dataset=pairs)
trainer.train()

## 13.9 Synthetic data and distillation: learning from a stronger model

A strong teacher model writes example answers and a small student model is fine-tuned on them. For small models this often beats training them with RL directly, which makes it the cheap route to a capable model.

[Read the lesson](https://aicourses.dev/research/fine-tuning/distillation)

### Soft targets: what the top answer hides

To show more of it, divide the logits by a **temperature** $T$ above 1 before the softmax, as in lesson 9.4: the distribution flattens, and the small probabilities grow enough to teach from. The student then minimises the KL divergence from the softened teacher to its own softened distribution, $D_{\text{KL}}(p_{\text{teacher}} \,\|\, q_{\text{student}})$, with the same direction as lesson 9.5.

In [ ]:
import torch
import torch.nn.functional as F

# next-token logits for "cat", "kitten", "dog", "car"
teacher = torch.tensor([4.0, 3.0, 1.0, -2.0])
student = torch.tensor([4.0, 0.0, 0.0, 0.0])
for T in (1.0, 2.0):
    p = F.softmax(teacher / T, dim=0)          # softened teacher
    log_q = F.log_softmax(student / T, dim=0)
    kl = F.kl_div(log_q, p, reduction="sum")   # KL(teacher || student)
    print(T, p.numpy().round(3), round(kl.item(), 3))
# 1.0 [0.704 0.259 0.035 0.002] 0.511
# 2.0 [0.532 0.323 0.119 0.026] 0.227

### Synthetic data you can trust

For the second route, the pipeline is: collect prompts, have the teacher write several answers to each, keep only good ones, clean them, and fine-tune. On a Colab GPU runtime, vLLM from lesson 12.2 can write the answers, with `n=4` in `SamplingParams` for four per prompt. When the task has checkable answers, such as maths, keep only the answers whose final result is correct. This is often called **rejection sampling**: sample, check, reject the failures.

In [ ]:
import re

gold = {"A tray holds 12 buns. How many in 5 trays?": "60",
        "Rolls cost 15 rupees. What do 4 cost?": "60"}
teacher = {  # four sampled teacher answers per question
    "A tray holds 12 buns. How many in 5 trays?":
        ["12 x 5 = 60. #### 60", "5 + 12 = 17. #### 17",
         "Five trays of 12 is 60. #### 60", "#### 60"],
    "Rolls cost 15 rupees. What do 4 cost?":
        ["15 x 4 = 60. #### 60", "15 x 4 = 45. #### 45",
         "4 rolls cost 60 rupees.", "15 x 4 = 60. #### 60"],
}

def final(text):
    found = re.findall(r"#### (-?\d+)", text)
    return found[-1] if found else None

keep = [(q, a) for q, answers in teacher.items() for a in answers
        if final(a) == gold[q] and len(a) > 10]   # right, shows work
print(len(keep), "of", sum(len(v) for v in teacher.values()), "kept")
# 4 of 8 kept